# BIL485 Derin Öğrenme — Ödev 2: Computer Vision
## CIFAR-10 Görüntü Sınıflandırma — ImageNet ön-eğitimli ResNet18 ile Transfer Öğrenme

**Kısa kullanım talimatı (Google Colab):**

1. Menüden **Runtime (Çalışma zamanı) > Change runtime type (Çalışma zamanı türünü değiştir)** → **T4 GPU** seç → Kaydet.
2. **Runtime > Run all (Tümünü çalıştır)** de. Toplam süre ücretsiz T4 üzerinde yaklaşık **15–20 dakika**.
3. Notebook "Gerçek fotoğraflarla test" bölümüne gelince bir **dosya yükleme (Choose Files) butonu** çıkar. Telefonla çektiğin **5 fotoğrafı** seç.
   - İpucu: Fotoğrafları yüklemeden önce `foto1.jpg … foto5.jpg` diye adlandırırsan etiket sözlüğü hazır şablona birebir oturur.
4. Yükleme sonrası `GERCEK_ETIKETLER` sözlüğüne her fotoğrafın gerçek sınıfını yaz ve **sadece o hücreyi** tekrar çalıştır → karşılaştırma tablosu çıkar.
5. Gradio hücresi bir **public link (…gradio.live)** üretir; tarayıcıda açıp kendi resmini yükleyerek en olası 3 sınıfı görebilirsin.
6. En sondaki **RAPOR ÖZETİ** bloğundaki değerleri rapor tablosuna kopyala; son hücre tüm sonuçları `sonuclar.zip` olarak indirir.

> Not: `FAST_TEST=1` ortam değişkeni sadece geliştirici testi içindir (sahte veri, CPU, çok küçük boyutlar). Colab'da hiçbir şey değiştirmene gerek yok.

## 1. Kurulum ve kütüphaneler
Colab'da `torch`, `torchvision`, `scikit-learn`, `matplotlib` hazır gelir; sadece `gradio` eksikse kurulur.

In [ ]:
import os, sys, time, json, random, shutil, subprocess, zipfile

FAST_TEST = os.environ.get("FAST_TEST", "0") == "1"   # sadece geliştirici testi
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Gradio yoksa kur (Colab'da birkaç saniye sürer)
try:
    import gradio as gr
except ImportError:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
        import gradio as gr
    else:
        gr = None
        print("Uyarı: gradio kurulu değil, Gradio bölümü atlanacak.")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Colab:", IN_COLAB, "| FAST_TEST:", FAST_TEST)

## 2. Ayarlar, sabit tohum (seed) ve cihaz
Tüm hiperparametreler tek yerde. Sabit `SEED` sayesinde train/val bölünmesi ve ağırlık başlangıcı her çalıştırmada aynı olur (tekrarlanabilirlik).

In [ ]:
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"            # karma hassasiyet (mixed precision) sadece GPU'da
torch.backends.cudnn.benchmark = True       # sabit giriş boyutunda en hızlı konvolüsyon algoritmasını seçer

# ---- Hiperparametreler ----
IMG_SIZE     = 128      # 32x32 CIFAR görüntüleri 128x128'e büyütülür
BATCH_SIZE   = 128
EPOCHS       = 4
LR           = 1e-3     # AdamW başlangıç öğrenme oranı
WEIGHT_DECAY = 0.05     # AdamW ağırlık çürümesi
VAL_SIZE     = 5000     # 50k eğitim verisinden ayrılan doğrulama örneği
NUM_WORKERS  = 2

if FAST_TEST:   # sadece hızlı mantık testi için küçük değerler
    IMG_SIZE, BATCH_SIZE, EPOCHS, VAL_SIZE, NUM_WORKERS = 64, 16, 1, 40, 0

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
CLASS_TR = {"airplane": "uçak", "automobile": "otomobil", "bird": "kuş", "cat": "kedi",
            "deer": "geyik", "dog": "köpek", "frog": "kurbağa", "horse": "at",
            "ship": "gemi", "truck": "kamyon"}
NUM_CLASSES = len(CLASS_NAMES)

OUT_DIR = "sonuclar"
os.makedirs(OUT_DIR, exist_ok=True)

if DEVICE.type == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
else:
    GPU_NAME = "CPU (GPU yok)"
print("Cihaz:", DEVICE, "|", GPU_NAME, "| AMP:", USE_AMP)

## 3. Veri seti: CIFAR-10 ve train / validation / test bölünmesi
- **CIFAR-10**: 10 sınıf, 32×32 renkli görüntü; 50.000 eğitim + 10.000 test. Kaynak: https://www.cs.toronto.edu/~kriz/cifar.html
- **Bölünme:** 50k eğitim verisi sabit tohumla **45.000 train / 5.000 validation** olarak karıştırılıp bölünür; resmi **10.000 test** seti en sona kadar hiç kullanılmaz.
- **Ön işleme:**
  - Train: `RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0))` + `RandomHorizontalFlip` (hafif veri artırma) → tensör → ImageNet normalizasyonu.
  - Val/Test: sadece `Resize(IMG_SIZE)` → tensör → ImageNet normalizasyonu (**artırma yok**, böylece ölçüm adil olur).
- Train ve val için **aynı veriye bakan iki ayrı dataset nesnesi** (farklı transform'larla) oluşturulur; `Subset` ile aynı indeksler üzerinden ayrılır.
- ImageNet ortalaması/std'si kullanılır çünkü ön-eğitimli ağırlıklar bu istatistiklerle normalize edilmiş girişlerle eğitildi.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

if not FAST_TEST:
    full_train_aug   = datasets.CIFAR10("data", train=True,  download=True, transform=train_tf)
    full_train_noaug = datasets.CIFAR10("data", train=True,  download=False, transform=eval_tf)
    test_set         = datasets.CIFAR10("data", train=False, download=True, transform=eval_tf)
else:
    # Geliştirici testi: internetsiz ortamda sahte 32x32, 10 sınıflı veri
    full_train_aug   = datasets.FakeData(200, (3, 32, 32), NUM_CLASSES, transform=train_tf, random_offset=0)
    full_train_noaug = datasets.FakeData(200, (3, 32, 32), NUM_CLASSES, transform=eval_tf,  random_offset=0)
    test_set         = datasets.FakeData(60,  (3, 32, 32), NUM_CLASSES, transform=eval_tf,  random_offset=10_000)

n_total = len(full_train_aug)
perm = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED)).tolist()
val_idx, train_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]
train_set = Subset(full_train_aug, train_idx)     # artırmalı
val_set   = Subset(full_train_noaug, val_idx)     # artırmasız
assert len(set(train_idx) & set(val_idx)) == 0    # sızıntı (leakage) yok

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                 persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_set, shuffle=True, drop_last=True,
                          generator=torch.Generator().manual_seed(SEED), **loader_kw)
val_loader   = DataLoader(val_set,  shuffle=False, **loader_kw)
test_loader  = DataLoader(test_set, shuffle=False, **loader_kw)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [ ]:
# Birkaç eğitim örneğini (artırma sonrası) görselleştir
def denorm(t):
    m = torch.tensor(IMAGENET_MEAN).view(3, 1, 1); s = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, i in zip(axes.flat, range(12)):
    img, lbl = train_set[i]
    ax.imshow(denorm(img)); ax.set_title(f"{CLASS_NAMES[lbl]}\n({CLASS_TR[CLASS_NAMES[lbl]]})", fontsize=9); ax.axis("off")
plt.suptitle("Eğitim örnekleri (veri artırma uygulanmış)"); plt.tight_layout(); plt.show()

## 4. Model: ImageNet ön-eğitimli ResNet18 (transfer öğrenme / fine-tuning)
- `torchvision.models.resnet18(weights=IMAGENET1K_V1)` yüklenir: 1,2 milyon ImageNet görüntüsünde öğrenilmiş kenar/doku/şekil filtreleri hazır gelir.
- Son tam bağlı katman `fc: 512 → 1000` yerine **`fc: 512 → 10`** ile değiştirilir.
- **Tüm katmanlar** eğitilir (fine-tuning). Ağ küçük (≈11,2 M parametre) olduğu için T4'te birkaç epokta hızlıca yüksek doğruluğa ulaşır.
- **Çıkış fonksiyonu:** Model 10 adet ham skor (logit) üretir; olasılık için **softmax** uygulanır. Eğitimde softmax ayrıca eklenmez, çünkü `CrossEntropyLoss` içinde `log_softmax` zaten var (sayısal olarak daha kararlı). Gizli katmanlarda ResNet'in standart **ReLU** aktivasyonu kullanılır.

In [ ]:
def build_model(pretrained=True):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    try:
        m = models.resnet18(weights=weights)
    except Exception as e:   # (sadece internetsiz test ortamı için yedek)
        if not FAST_TEST:
            raise
        print("Ön-eğitimli ağırlık indirilemedi, test modunda weights=None:", type(e).__name__)
        m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

model = build_model(pretrained=True).to(DEVICE)
model = model.to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parametre sayısı: {n_params/1e6:.2f} M")
print(model.fc)

## 5. Eğitim ayarları ve gerekçeleri

| Ayar | Değer | Gerekçe |
|---|---|---|
| Giriş boyutu | 128×128×3 | Ön-eğitimli ResNet daha büyük görüntülerde eğitildi; 32 px'te ilk katmanlar (stride'lı conv + maxpool) görüntüyü çok küçültür. 128 px doğruluk/hız dengesi için iyi bir orta nokta. |
| Batch size | 128 | T4'ün 16 GB belleğine rahat sığar, AMP ile GPU'yu verimli kullanır; gradyan tahmini yeterince kararlı. |
| Epoch | 4 | Ön-eğitimli ağ hızlı yakınsar; 15–20 dk Colab bütçesine sığar. |
| Optimizer | AdamW (lr=1e-3, weight_decay=0.05) | Adam'ın uyarlanabilir adımları hızlı yakınsama sağlar; AdamW ağırlık çürümesini gradyandan ayırarak (decoupled) daha doğru düzenlileştirme yapar. |
| LR planı | Cosine annealing (her adımda, 1e-3 → ~0) | Başta büyük adımlarla hızlı ilerleyip sona doğru küçük adımlarla ince ayar yapar; sabit LR'ye göre genelde daha iyi son doğruluk. |
| Kayıp fonksiyonu | CrossEntropyLoss | Çok sınıflı, tek etiketli sınıflandırma için standart; softmax + negatif log-olabilirlik birleşimidir. |
| Çıkış fonksiyonu | Softmax (logitler üzerinde) | 10 sınıf için toplamı 1 olan olasılık dağılımı verir; top-3 tahmin ve güven skoru buradan gelir. |
| Karma hassasiyet | torch.amp (float16) | T4 Tensor Core'larını kullanır, eğitimi ~2 kat hızlandırır, belleği azaltır. |
| Model seçimi | En iyi val accuracy → `best_model.pth` | Test setine bakmadan, doğrulama setine göre en iyi epok seçilir. |

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

def run_epoch(model, loader, train=True):
    model.train(train)
    tot_loss, tot_correct, tot_n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            tot_loss += loss.item() * y.size(0)
            tot_correct += (logits.argmax(1) == y).sum().item()
            tot_n += y.size(0)
    return tot_loss / tot_n, tot_correct / tot_n

## 6. Eğitim
Her epokta train ve val kaybı/doğruluğu kaydedilir; val doğruluğu en iyi olan model `best_model.pth` olarak saklanır. Toplam eğitim süresi ölçülür.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
best_val_acc, best_epoch = -1.0, -1
BEST_PATH = "best_model.pth"

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
train_start = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, train=True)
    va_loss, va_acc = run_epoch(model, val_loader, train=False)
    ep_time = time.time() - t0
    for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc, ep_time]):
        history[k].append(v)
    flag = ""
    if va_acc > best_val_acc:
        best_val_acc, best_epoch = va_acc, epoch
        torch.save({"state_dict": model.state_dict(), "class_names": CLASS_NAMES,
                    "img_size": IMG_SIZE, "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
                    "arch": "resnet18", "epoch": epoch, "val_acc": va_acc}, BEST_PATH)
        flag = "  <- en iyi, kaydedildi"
    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | {ep_time:.1f} s{flag}")
TRAIN_TIME_S = time.time() - train_start
print(f"\nToplam eğitim süresi: {TRAIN_TIME_S/60:.2f} dk ({TRAIN_TIME_S:.1f} s)")
print(f"En iyi val accuracy: {best_val_acc:.4f} (epoch {best_epoch})")

In [ ]:
# Öğrenme eğrileri (learning curves)
ep = range(1, EPOCHS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, history["train_loss"], "o-", label="Train"); a1.plot(ep, history["val_loss"], "s-", label="Validation")
a1.set_title("Kayıp (CrossEntropy)"); a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.legend(); a1.grid(alpha=.3)
a2.plot(ep, history["train_acc"], "o-", label="Train"); a2.plot(ep, history["val_acc"], "s-", label="Validation")
a2.set_title("Doğruluk (Accuracy)"); a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy"); a2.legend(); a2.grid(alpha=.3)
for a in (a1, a2):
    a.set_xticks(list(ep))
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "learning_curves.png"), dpi=150); plt.show()

## 7. Test seti değerlendirmesi
En iyi model (val'e göre) yüklenir ve **hiç görülmemiş 10.000 test görüntüsünde** ölçülür.
- **Accuracy:** doğru tahmin / toplam.
- **Precision (kesinlik):** "X dediğinde ne kadar haklı?" = TP / (TP + FP).
- **Recall (duyarlılık):** "Gerçek X'lerin ne kadarını yakaladı?" = TP / (TP + FN).
- **F1:** precision ile recall'un harmonik ortalaması = 2·P·R / (P + R).
- **Macro** ortalama: her sınıfın skoru eşit ağırlıkla ortalanır (CIFAR-10 test seti dengeli: sınıf başına 1000).

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

@torch.no_grad()
def predict_loader(model, loader):
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        preds.append(logits.argmax(1).cpu()); labels.append(y)
    return torch.cat(labels).numpy(), torch.cat(preds).numpy()

y_true, y_pred = predict_loader(model, test_loader)
labels_all = list(range(NUM_CLASSES))
test_acc = accuracy_score(y_true, y_pred)
p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(y_true, y_pred, average="macro",
                                                          labels=labels_all, zero_division=0)
p_c, r_c, f1_c, sup_c = precision_recall_fscore_support(y_true, y_pred, average=None,
                                                        labels=labels_all, zero_division=0)

print(f"TEST  Accuracy: {test_acc:.4f} | Precision(macro): {p_mac:.4f} | "
      f"Recall(macro): {r_mac:.4f} | F1(macro): {f1_mac:.4f}\n")
per_class = pd.DataFrame({"sınıf": CLASS_NAMES, "Türkçe": [CLASS_TR[c] for c in CLASS_NAMES],
                          "precision": p_c, "recall": r_c, "f1": f1_c, "destek": sup_c}).round(4)
from IPython.display import display
display(per_class)
per_class.to_csv(os.path.join(OUT_DIR, "sinif_bazli_metrikler.csv"), index=False)
print(classification_report(y_true, y_pred, labels=labels_all, target_names=CLASS_NAMES,
                            digits=4, zero_division=0))

In [ ]:
# Karışıklık matrisi (confusion matrix)
cm = confusion_matrix(y_true, y_pred, labels=labels_all)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(labels_all); ax.set_yticks(labels_all)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Tahmin edilen sınıf"); ax.set_ylabel("Gerçek sınıf")
ax.set_title(f"Karışıklık Matrisi — Test (accuracy={test_acc:.4f})")
thr = cm.max() / 2 if cm.max() > 0 else 0.5
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > thr else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

# En çok karıştırılan sınıf çiftleri
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]
print("En çok karıştırılan çiftler (gerçek -> tahmin: adet):")
for i, j in top:
    print(f"  {CLASS_NAMES[i]} -> {CLASS_NAMES[j]}: {off[i, j]}")

## 8. Çıkarım (inference) süresi
- **Gecikme (latency):** batch=1, tek görüntü için ortalama süre (ms/görüntü). Önce ısınma (warm-up) yapılır, GPU'da `torch.cuda.synchronize()` ile doğru ölçülür. Sadece model ileri geçişi (forward) ölçülür; ön işleme dahil değildir.
- **Verim (throughput):** batch=256 ile saniyede işlenen görüntü sayısı.

In [ ]:
@torch.no_grad()
def time_inference(model, batch, n_iter, n_warmup=10):
    x = torch.randn(batch, 3, IMG_SIZE, IMG_SIZE, device=DEVICE).to(memory_format=torch.channels_last)
    def fwd():
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            model(x)
    for _ in range(n_warmup):
        fwd()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        fwd()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter   # saniye / batch

model.eval()
n1, nb, TP_BATCH = (5, 2, 8) if FAST_TEST else (200, 20, 256)
lat_s = time_inference(model, 1, n1, n_warmup=2 if FAST_TEST else 20)
LATENCY_MS = lat_s * 1000
tb_s = time_inference(model, TP_BATCH, nb, n_warmup=2 if FAST_TEST else 5)
THROUGHPUT = TP_BATCH / tb_s
print(f"Gecikme (batch=1): {LATENCY_MS:.2f} ms/görüntü")
print(f"Verim (batch={TP_BATCH}): {THROUGHPUT:.0f} görüntü/s  ({1000/THROUGHPUT:.3f} ms/görüntü)")

## 9. Gerçek fotoğraflarla test (telefon fotoğrafları)
Aşağıdaki hücre bir **yükleme butonu** açar: telefonla çektiğin **5 fotoğrafı** seç (CIFAR-10 sınıflarından: uçak, otomobil, kuş, kedi, geyik, köpek, kurbağa, at, gemi, kamyon).
Her fotoğraf için en olası sınıf ve güven (softmax olasılığı) gösterilir, grid `phone_test_grid.png` olarak kaydedilir.

In [ ]:
PHOTO_DIR = "telefon_fotograflari"
os.makedirs(PHOTO_DIR, exist_ok=True)

if IN_COLAB:
    from google.colab import files
    print("5 fotoğrafı seç (Choose Files):")
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(PHOTO_DIR, name), "wb") as f:
            f.write(data)
elif FAST_TEST:
    # Test ortamı: rastgele 5 sahte "fotoğraf" üret
    rng = np.random.default_rng(SEED)
    for k in range(1, 6):
        Image.fromarray(rng.integers(0, 255, (480, 640, 3), dtype=np.uint8)).save(
            os.path.join(PHOTO_DIR, f"foto{k}.jpg"))
else:
    print(f"Colab dışında: fotoğrafları '{PHOTO_DIR}/' klasörüne elle koy.")

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
photo_files = sorted(f for f in os.listdir(PHOTO_DIR) if f.lower().endswith(IMG_EXT))
print("Bulunan fotoğraflar:", photo_files)

@torch.no_grad()
def predict_pil(img, k=3):
    # PIL görüntü -> (top-k sınıf adları, olasılıklar)
    img = img.convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(DEVICE).to(memory_format=torch.channels_last)
    model.eval()
    probs = F.softmax(model(x).float(), dim=1)[0].cpu()
    p, idx = probs.topk(k)
    return [CLASS_NAMES[i] for i in idx.tolist()], p.tolist()

phone_preds = {}
if photo_files:
    n = len(photo_files); cols = min(n, 5); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.8 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, fname in zip(axes.flat, photo_files):
        from PIL import ImageOps
        img = ImageOps.exif_transpose(Image.open(os.path.join(PHOTO_DIR, fname)))  # telefon döndürmesini düzelt
        names, probs = predict_pil(img)
        phone_preds[fname] = (names, probs)
        ax.imshow(img.convert("RGB"))
        ax.set_title(f"{fname}\nTahmin: {names[0]} ({CLASS_TR[names[0]]})\nGüven: %{probs[0]*100:.1f}", fontsize=9)
        print(f"{fname}: " + ", ".join(f"{c} %{p*100:.1f}" for c, p in zip(names, probs)))
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "phone_test_grid.png"), dpi=150); plt.show()
else:
    print("Fotoğraf yüklenmedi; bu hücreyi tekrar çalıştırıp fotoğraf seçebilirsin.")

### Gerçek etiketleri gir
Aşağıdaki sözlüğe her fotoğrafın **gerçek** sınıfını yaz: İngilizce (`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`) ya da Türkçe (`uçak, otomobil, kuş, kedi, geyik, köpek, kurbağa, at, gemi, kamyon`) ve **sadece bu hücreyi** tekrar çalıştır.
Dosya adların farklıysa anahtarları (soldaki adları) yukarıda yazdırılan dosya adlarıyla değiştir. `"?"` kalan satırlar "etiket girilmedi" sayılır.

In [ ]:
GERCEK_ETIKETLER = {
    "foto1.jpg": "?",   # örn. "cat"
    "foto2.jpg": "?",   # örn. "dog"
    "foto3.jpg": "?",   # örn. "automobile"
    "foto4.jpg": "?",   # örn. "bird"
    "foto5.jpg": "?",   # örn. "truck"
}

TR_TO_EN = {v: k for k, v in CLASS_TR.items()}
rows = []
for fname, (names, probs) in phone_preds.items():
    gercek = GERCEK_ETIKETLER.get(fname, "?").strip().lower()
    gercek = TR_TO_EN.get(gercek, gercek)          # Türkçe yazıldıysa İngilizceye çevir
    if gercek not in CLASS_NAMES and gercek != "?":
        print(f"Uyarı: '{gercek}' geçerli bir CIFAR-10 sınıfı değil ({fname})")
    dogru = "-" if gercek == "?" else ("EVET" if gercek == names[0] else "HAYIR")
    top3 = "EVET" if gercek in names else ("-" if gercek == "?" else "HAYIR")
    rows.append({"fotoğraf": fname, "gerçek": gercek, "tahmin": names[0],
                 "güven (%)": round(probs[0] * 100, 1), "doğru mu?": dogru,
                 "top-3 içinde mi?": top3,
                 "top-3": ", ".join(f"{c}:{p*100:.0f}%" for c, p in zip(names, probs))})
phone_df = pd.DataFrame(rows)
print(phone_df.to_string(index=False))
etiketli = phone_df[phone_df["gerçek"] != "?"] if len(phone_df) else phone_df
if len(etiketli):
    PHONE_ACC = f"{(etiketli['doğru mu?'] == 'EVET').sum()}/{len(etiketli)}"
    print(f"\nTelefon fotoğraflarında top-1 doğru: {PHONE_ACC}")
else:
    PHONE_ACC = "etiket girilmedi"
    print("\nGerçek etiketler henüz girilmedi (GERCEK_ETIKETLER sözlüğünü doldur).")
if len(phone_df):
    phone_df.to_csv(os.path.join(OUT_DIR, "telefon_testi.csv"), index=False)

### Sonuçlar nasıl yorumlanır? (Alan kayması / domain shift)
Test setinde yüksek doğruluk alan model, telefon fotoğraflarında bazen yanılabilir. Bunun ana nedeni **alan kayması (domain shift)**: modelin eğitildiği veri dağılımı ile gerçek fotoğrafların dağılımı farklıdır.

- **Çözünürlük:** CIFAR-10 görüntüleri **32×32** piksel; model aslında bulanık, düşük ayrıntılı görüntüler görerek öğrendi (128'e büyütülmüş olsa da bilgi 32 px'lik). Telefon fotoğrafı ise yüksek çözünürlüklü; ince dokular modele yabancı gelebilir.
- **Kadraj / ortalanmış nesne:** CIFAR'da nesne genelde **ortada ve kareyi dolduruyor**. Nesne küçükse, kenardaysa veya arka plan kalabalıksa tahmin kötüleşir. Ayrıca fotoğraf kareye sıkıştırıldığı (`Resize`) için en-boy oranı bozulur.
- **Bakış açısı, ışık, arka plan:** Gece çekimi, alışılmadık açı, kısmen görünen nesne hata riskini artırır.
- **Kapalı küme (closed-set) varsayımı:** Model **her zaman** 10 sınıftan birini seçer. Bu sınıflardan biri olmayan bir şey (örn. kalem) çekersen yine bir sınıf söyler; bu "hata" değil, modelin tasarım sınırıdır.
- **Benzer sınıflar:** kedi↔köpek, otomobil↔kamyon, geyik↔at gibi çiftler karışıklık matrisinde de en çok karışan sınıflardır; telefon testinde de en olası hatalar bunlardır.
- **Güven skoru:** Yanlış tahminin güveni düşükse (örn. %40) model "emin değil" demektir; yanlış ama yüksek güvenli (örn. %95) tahminler, modelin aşırı güvenli (overconfident) olabileceğini gösterir.
- **Doğru tahminler** ise öğrenilen özelliklerin gerçek dünyaya da genellenebildiğini gösterir (özellikle nesne ortada, net ve kadrajı dolduruyorsa).

**Daha iyi sonuç için ipuçları:** nesneyi ortala, yakından çek, sade arka plan kullan; fotoğrafı kare kırp.

## 10. Dağıtım (deployment): Gradio arayüzü
Kullanıcı bir görüntü yükler, model **en olası 3 sınıfı** olasılıklarıyla gösterir. Colab'da `share=True` ile herkese açık geçici bir link (`…gradio.live`, ~72 saat geçerli) oluşur.
Aynı uygulamanın bağımsız sürümü GitHub reposundaki `app.py` dosyasıdır.

In [ ]:
def gradio_predict(img):
    if img is None:
        return {}
    names, probs = predict_pil(img, k=3)
    return {f"{c} ({CLASS_TR[c]})": float(p) for c, p in zip(names, probs)}

demo = None
if gr is not None:
    gr_major = int(gr.__version__.split(".")[0])
    flag_kw = {"flagging_mode": "never"} if gr_major >= 5 else {"allow_flagging": "never"}
    demo = gr.Interface(
        fn=gradio_predict,
        inputs=gr.Image(type="pil", label="Görüntü yükle"),
        outputs=gr.Label(num_top_classes=3, label="En olası 3 sınıf"),
        title="CIFAR-10 Sınıflandırıcı — ResNet18 (Transfer Öğrenme)",
        description=("Bir görüntü yükleyin; model 10 CIFAR-10 sınıfından en olası 3 tanesini olasılıklarıyla gösterir. "
                     "Sınıflar: uçak, otomobil, kuş, kedi, geyik, köpek, kurbağa, at, gemi, kamyon."),
        **flag_kw,
    )
    if IN_COLAB:
        demo.launch(share=True, debug=False)   # public link aşağıda görünür
    else:
        print("Gradio arayüzü oluşturuldu (Colab dışında otomatik başlatılmadı).")

## 11. RAPOR ÖZETİ
Aşağıdaki bloktaki değerleri rapor şablonundaki tabloya kopyala. Aynı içerik `sonuclar/rapor_ozeti.txt` ve `sonuclar/metrikler.json` dosyalarına da kaydedilir.

In [ ]:
best_idx = best_epoch - 1
summary = {
    "Project Title": "CIFAR-10 Görüntü Sınıflandırma: ImageNet Ön-Eğitimli ResNet18 ile Transfer Öğrenme",
    "Problem/Task": "Çok sınıflı görüntü sınıflandırma (10 sınıf)",
    "Dataset": f"CIFAR-10 (60.000 adet 32x32 RGB; split: {len(train_set)} train / {len(val_set)} val / {len(test_set)} test, seed={SEED})",
    "Dataset URL": "https://www.cs.toronto.edu/~kriz/cifar.html",
    "Model Architecture": f"ResNet18 (torchvision, ImageNet1K_V1 ön-eğitimli), fc 512->10, tüm katmanlar fine-tune, {n_params/1e6:.2f}M parametre",
    "Input Size": f"{IMG_SIZE}x{IMG_SIZE}x3 (32x32'den büyütülmüş, ImageNet normalizasyonu)",
    "Hyperparameters": f"lr={LR}, batch size={BATCH_SIZE}, epochs={EPOCHS}, weight decay={WEIGHT_DECAY}, cosine LR schedule, AMP (fp16), augment: RandomResizedCrop(0.7-1)+HFlip",
    "Optimizer": f"AdamW (lr={LR}, weight_decay={WEIGHT_DECAY})",
    "Activation/Output Function": "Gizli katmanlar ReLU; çıkış: logitler üzerinde Softmax",
    "Loss Function": "CrossEntropyLoss",
    "Evaluation Metrics": "Accuracy, Precision, Recall, F1 (macro + sınıf bazlı), Confusion Matrix, öğrenme eğrileri, eğitim/çıkarım süresi",
    "Training Environment": f"Google Colab, {GPU_NAME}, PyTorch {torch.__version__}",
    "Best Performance": (f"Test Acc={test_acc:.4f}, Precision(macro)={p_mac:.4f}, Recall(macro)={r_mac:.4f}, "
                         f"F1(macro)={f1_mac:.4f} | en iyi Val Acc={best_val_acc:.4f} (epoch {best_epoch})"),
    "Inference Time": f"{LATENCY_MS:.2f} ms/görüntü (batch=1), throughput {THROUGHPUT:.0f} görüntü/s (batch={TP_BATCH})",
    "Deployment Technology": "Gradio (Colab'da share=True public link; app.py ile yerel)",
    "Repository": "<GitHub repo linkini buraya yaz>",
}
extra = {
    "Eğitim süresi": f"{TRAIN_TIME_S/60:.2f} dk ({TRAIN_TIME_S:.0f} s, {EPOCHS} epoch, ort. {np.mean(history['epoch_time']):.0f} s/epoch)",
    "Son epoch train acc / loss": f"{history['train_acc'][-1]:.4f} / {history['train_loss'][-1]:.4f}",
    "En iyi epoch val loss": f"{history['val_loss'][best_idx]:.4f}",
    "Telefon testi (top-1)": PHONE_ACC,
}
lines = ["=" * 78, "RAPOR ÖZETİ".center(78), "=" * 78]
lines += [f"{k:<28}: {v}" for k, v in summary.items()]
lines += ["-" * 78] + [f"{k:<28}: {v}" for k, v in extra.items()]
lines += ["-" * 78, "Sınıf bazlı metrikler (test):", per_class.to_string(index=False), "=" * 78]
report_text = "\n".join(lines)
print(report_text)

with open(os.path.join(OUT_DIR, "rapor_ozeti.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
with open(os.path.join(OUT_DIR, "metrikler.json"), "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "extra": extra, "history": history,
               "test": {"accuracy": test_acc, "precision_macro": p_mac, "recall_macro": r_mac, "f1_macro": f1_mac},
               "latency_ms_batch1": LATENCY_MS, "throughput_img_s": THROUGHPUT,
               "train_time_s": TRAIN_TIME_S}, f, ensure_ascii=False, indent=2)

## 12. Sonuçları indir
Grafikler, metrikler, rapor özeti, telefon testi ve `best_model.pth` tek bir `sonuclar.zip` dosyasında toplanır (GitHub reposuna `app.py` ile birlikte koymak için).

In [ ]:
if os.path.exists(BEST_PATH):
    shutil.copy(BEST_PATH, os.path.join(OUT_DIR, BEST_PATH))
zip_path = "sonuclar.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fs in os.walk(OUT_DIR):
        for fn in fs:
            zf.write(os.path.join(root, fn))
print("Oluşturuldu:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("İçerik:", sorted(os.listdir(OUT_DIR)))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)